# KnowledgeHub — Indexing & Search Demo

This notebook demonstrates the **document indexing and keyword-search algorithm** (Q5) at the heart of KnowledgeHub, then drives the full REST API end-to-end.

It needs **no external services** — the engine falls back to a pure-Python inverted index and SQLite, so everything runs in-process.

**Sections**
1. Tokenization & analysis
2. Building the inverted index
3. TF-IDF + cosine ranking
4. Inspecting the index internals
5. Full API walkthrough (auth → docs → versioning → permissions → search → collaboration)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # import the `app` package
from app.services.inverted_index import InvertedIndex, tokenize, STOP_WORDS

## 1. Tokenization & analysis

Every document and query passes through the same analyzer: lower-case, split on word boundaries, drop common stop-words. Applying identical analysis to documents and queries is what makes term matching consistent.

In [2]:
sample = 'The Kubernetes Production Deployment Guide (v2.0)!'
print('raw     :', sample)
print('tokens  :', tokenize(sample))
print('stopwords removed e.g.:', sorted(list(STOP_WORDS))[:8], '...')

raw     : The Kubernetes Production Deployment Guide (v2.0)!
tokens  : ['kubernetes', 'production', 'deployment', 'guide', 'v2', '0']
stopwords removed e.g.: ['a', 'an', 'and', 'are', 'as', 'at', 'be', 'but'] ...


## 2. Building the inverted index

We index a small corpus of knowledge-base articles. Internally the index stores a **postings map**: `term -> {doc_id: term_frequency}`.

In [3]:
corpus = {
    'd1': ('Kubernetes Production Deployment Guide',
           'How to deploy services to production Kubernetes clusters with rolling updates and canary releases.'),
    'd2': ('Disaster Recovery Policy',
           'Backup schedules, RPO and RTO targets, and the database restore procedure for production.'),
    'd3': ('Onboarding FAQ',
           'Frequently asked questions for new engineers joining the platform team.'),
    'd4': ('Incident Response Runbook',
           'Steps to triage a production incident, escalate, and run a postmortem.'),
}

index = InvertedIndex()
for doc_id, (title, body) in corpus.items():
    index.add(doc_id, f'{title} {body}', title=title)

print('documents indexed:', index.num_docs)
print('unique terms     :', len(index.postings))

documents indexed: 4
unique terms     : 41


## 3. TF-IDF + cosine ranking

At query time each term is weighted by **TF-IDF** and documents are ranked by **cosine similarity** between the query and document vectors. Cosine normalisation prevents long documents from dominating purely because they contain more words.

In [4]:
def show(query):
    print(f'\nQuery: {query!r}')
    results = index.search(query)
    if not results:
        print('  (no matches)')
    for doc_id, score, snippet in results:
        print(f'  {score:.4f}  {index.title_of(doc_id):38}  {snippet}')

show('production deployment')
show('database backup recovery')
show('incident postmortem')
show('vacation policy')  # no relevant doc


Query: 'production deployment'
  0.3538  Kubernetes Production Deployment Guide  Kubernetes Production Deployment Guide How to deploy …
  0.0730  Disaster Recovery Policy                … and the database restore procedure for production.
  0.0730  Incident Response Runbook               … Response Runbook Steps to triage a production incident, escalate, and run a …

Query: 'database backup recovery'
  0.5158  Disaster Recovery Policy                Disaster Recovery Policy Backup schedules, RPO and …

Query: 'incident postmortem'
  0.6317  Incident Response Runbook               Incident Response Runbook Steps to triage …

Query: 'vacation policy'
  0.2978  Disaster Recovery Policy                Disaster Recovery Policy Backup schedules, RPO and RTO …


## 4. Inspecting the index internals

We can look directly at the postings list and the computed IDF weights to see *why* a document ranks where it does. Rarer terms carry more weight.

In [5]:
term = 'production'
print(f'postings for {term!r}:', dict(index.postings[term]))
print(f'IDF({term!r})      = {index._idf(term):.4f}')
print(f'IDF(\'kubernetes\') = {index._idf("kubernetes"):.4f}  # rarer -> higher weight')

postings for 'production': {'d1': 2, 'd2': 1, 'd4': 1}
IDF('production')      = 0.8473
IDF('kubernetes') = 1.6094  # rarer -> higher weight


## 5. Full API walkthrough

Now the same indexing logic running behind the **FastAPI** service, exercised with `TestClient` (no server process needed). This mirrors `smoke_test.py`.

In [6]:
os.environ['DATABASE_URL'] = 'sqlite:///./demo_nb.db'
if os.path.exists('demo_nb.db'):
    os.remove('demo_nb.db')
from fastapi.testclient import TestClient
from app.main import app
client = TestClient(app)
client.get('/health').json()

/Users/pranavkale/Desktop/system design sem4/knowledgehub/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
Redis unavailable (No module named 'redis'); using in-memory cache fallback


Elasticsearch unavailable (No module named 'elasticsearch'); using local index


INFO:httpx:HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


{'status': 'ok',
 'search_backend': 'inverted-index',
 'cache_backend': '_MemoryCache'}

In [7]:
# Register an author and a teammate
client.post('/auth/register', json={'email':'alice@corp.com','full_name':'Alice','password':'secret1','role':'editor'})
client.post('/auth/register', json={'email':'bob@corp.com','full_name':'Bob','password':'secret1','role':'viewer'})
alice = client.post('/auth/login', data={'username':'alice@corp.com','password':'secret1'}).json()['access_token']
bob   = client.post('/auth/login', data={'username':'bob@corp.com','password':'secret1'}).json()['access_token']
A = {'Authorization': f'Bearer {alice}'}
B = {'Authorization': f'Bearer {bob}'}
bob_id = client.get('/auth/me', headers=B).json()['id']
print('logged in as alice & bob')

INFO:httpx:HTTP Request: POST http://testserver/auth/register "HTTP/1.1 201 Created"


INFO:httpx:HTTP Request: POST http://testserver/auth/register "HTTP/1.1 201 Created"


INFO:httpx:HTTP Request: POST http://testserver/auth/login "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST http://testserver/auth/login "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://testserver/auth/me "HTTP/1.1 200 OK"


logged in as alice & bob


In [8]:
# Alice creates documents
for title, content, tags in [
    ('Kubernetes Production Deployment Guide','Deploy services to production Kubernetes clusters with rolling updates.',['devops']),
    ('Disaster Recovery Policy','Backup schedules, RPO/RTO targets and the database restore procedure.',['policy']),
    ('Onboarding FAQ','Frequently asked questions for new engineers.',['hr']),
]:
    client.post('/documents', json={'title':title,'content':content,'tags':tags}, headers=A)
doc = client.get('/search', params={'q':'kubernetes'}, headers=A).json()['hits'][0]
doc_id = doc['document_id']
print('created; top hit for kubernetes:', doc['title'])

INFO:httpx:HTTP Request: POST http://testserver/documents "HTTP/1.1 201 Created"


INFO:httpx:HTTP Request: POST http://testserver/documents "HTTP/1.1 201 Created"


INFO:httpx:HTTP Request: POST http://testserver/documents "HTTP/1.1 201 Created"


INFO:httpx:HTTP Request: GET http://testserver/search?q=kubernetes "HTTP/1.1 200 OK"


created; top hit for kubernetes: Kubernetes Production Deployment Guide


In [9]:
# Versioning: edit creates a new immutable revision
client.put(f'/documents/{doc_id}', json={'content':'Updated: adds blue-green and canary strategies.'}, headers=A)
versions = client.get(f'/documents/{doc_id}/versions', headers=A).json()
print('versions (latest first):', [v['version'] for v in versions])

INFO:httpx:HTTP Request: PUT http://testserver/documents/17850e8b-db7e-4dbf-ba9d-667113eb14ed "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://testserver/documents/17850e8b-db7e-4dbf-ba9d-667113eb14ed/versions "HTTP/1.1 200 OK"


versions (latest first): [2, 1]


In [10]:
# Authorization: Bob is blocked until the doc is shared with him
print('bob before share ->', client.get(f'/documents/{doc_id}', headers=B).status_code)
client.post(f'/documents/{doc_id}/share', json={'user_id':bob_id,'level':'viewer'}, headers=A)
print('bob after  share ->', client.get(f'/documents/{doc_id}', headers=B).status_code)

INFO:httpx:HTTP Request: GET http://testserver/documents/17850e8b-db7e-4dbf-ba9d-667113eb14ed "HTTP/1.1 403 Forbidden"


INFO:httpx:HTTP Request: POST http://testserver/documents/17850e8b-db7e-4dbf-ba9d-667113eb14ed/share "HTTP/1.1 204 No Content"


INFO:httpx:HTTP Request: GET http://testserver/documents/17850e8b-db7e-4dbf-ba9d-667113eb14ed "HTTP/1.1 200 OK"


bob before share -> 403
bob after  share -> 200


In [11]:
# Permission-filtered search: Bob only sees what he can access
print('alice sees:', [h['title'] for h in client.get('/search', params={'q':'guide policy faq'}, headers=A).json()['hits']])
print('bob   sees:', [h['title'] for h in client.get('/search', params={'q':'guide policy faq'}, headers=B).json()['hits']])

INFO:httpx:HTTP Request: GET http://testserver/search?q=guide+policy+faq "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://testserver/search?q=guide+policy+faq "HTTP/1.1 200 OK"


alice sees: ['Onboarding FAQ', 'Kubernetes Production Deployment Guide', 'Disaster Recovery Policy']
bob   sees: ['Kubernetes Production Deployment Guide']


In [12]:
# Caching: identical repeated query is served from cache
r1 = client.get('/search', params={'q':'production'}, headers=A).json()
r2 = client.get('/search', params={'q':'production'}, headers=A).json()
print('first call cached:', r1['cached'], '| second call cached:', r2['cached'])

INFO:httpx:HTTP Request: GET http://testserver/search?q=production "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://testserver/search?q=production "HTTP/1.1 200 OK"


first call cached: False | second call cached: True


In [13]:
# Collaboration: comment + activity feed
client.post(f'/documents/{doc_id}/comments', json={'body':'Should we add a Helm section?'}, headers=B)
feed = client.get(f'/documents/{doc_id}/activity', headers=A).json()
for e in feed:
    print(f"  {e['action']:10} {e['detail']}")

INFO:httpx:HTTP Request: POST http://testserver/documents/17850e8b-db7e-4dbf-ba9d-667113eb14ed/comments "HTTP/1.1 201 Created"


INFO:httpx:HTTP Request: GET http://testserver/documents/17850e8b-db7e-4dbf-ba9d-667113eb14ed/activity "HTTP/1.1 200 OK"


  commented  Should we add a Helm section?
  shared     4d2f14ea-5f9f-47a2-94c1-2f371781a5ba:viewer
  updated    v2
  created    Created 'Kubernetes Production Deployment Guide'


## Summary

We demonstrated the full lifecycle: **tokenize → index → TF-IDF/cosine rank**, then the same engine behind a secured API with versioning, per-document permissions, permission-filtered + cached search, and collaboration logging — the core of the KnowledgeHub design.